# Reproducibility smoke test on the free sample

This notebook runs `../validation/benchmark_script.py`, **unmodified and identical to the script used for the published benchmark**, against the free sample shipped in `../sample/`.

**Purpose:** show that the code runs end to end against the real delivered schema.

**Not the purpose:** reproduce the published metric. **The benchmark script cannot be run on this sample, and this notebook shows why instead of pretending otherwise.** The script reads `expression_profiles.parquet` (twelve expression features: SLC9A1, SLC16A1, SLC16A3, CA9, CA12, ATP6V1A, SLC4A4, SLC4A7, SLC7A11, G6PD, GCLC, glyco_score) and `screen_outcomes.parquet`. The free sample ships a single joined outcomes table with the kinetic model parameters and no expression columns, so there is nothing for the script to train on.

The number to cite is the one measured on the full dataset: AUC 0.9209-0.9985 across 3 conditions x 2 models (5-fold CV, seed 1337, full 10,123-profile dataset).


In [1]:
import sys, subprocess, shutil, tempfile, pathlib
import pandas as pd
HERE = pathlib.Path.cwd()
SAMPLE = (HERE / ".." / "sample").resolve()
SCRIPT = (HERE / ".." / "validation" / "benchmark_script.py").resolve()
work = pathlib.Path(tempfile.mkdtemp(prefix="smoke_"))   # all outputs go here, nothing is written into the repository
print("sample dir :", SAMPLE.name)
print("script     :", SCRIPT.name, "(unmodified copy of the published benchmark script)")
def clean(text):
    return text.replace(str(work), "<tmp>").replace(str(SAMPLE), "<sample>")


sample dir : sample
script     : benchmark_script.py (unmodified copy of the published benchmark script)


In [2]:
import re, json
script_text = SCRIPT.read_text()
features = re.search(r"FEATURES = \[(.*?)\]", script_text, re.S).group(1)
features = re.findall(r'"([^"]+)"', features)
sample_outcomes = pd.read_parquet(SAMPLE / "sample_screen_outcomes.parquet")
print("sample table   :", sample_outcomes.shape)
print("script needs   :", len(features), "expression feature columns from expression_profiles.parquet")
print("present in sample columns:", [f for f in features if f in sample_outcomes.columns])
print("missing from sample      :", [f for f in features if f not in sample_outcomes.columns])
print("expression_profiles.parquet in sample folder:", (SAMPLE / "expression_profiles.parquet").exists())

sample table   : (336, 25)
script needs   : 12 expression feature columns from expression_profiles.parquet
present in sample columns: []
missing from sample      : ['SLC9A1', 'SLC16A1', 'SLC16A3', 'CA9', 'CA12', 'ATP6V1A', 'SLC4A4', 'SLC4A7', 'SLC7A11', 'G6PD', 'GCLC', 'glyco_score']
expression_profiles.parquet in sample folder: False


## Reading the output

The check above is computed from the files: none of the twelve features the script trains on exist in the sample, and `expression_profiles.parquet` is not shipped. What the sample is good for is the outcome layer, which the next cells inspect.

In [3]:
outcomes = pd.read_parquet(SAMPLE / "sample_screen_outcomes.parquet")
tab = (outcomes.groupby(["cohort", "condition"]).dead.agg(rows="size", dead="sum")
       .assign(death_rate=lambda t: (t.dead / t.rows).round(3)))
print(outcomes.data_type.unique())
tab

<ArrowStringArray>
['in_silico_model_prediction_NOT_experimental']
Length: 1, dtype: str


rows  dead  death_rate
cohort         condition                          
normal_virtual control        20     0       0.000
               triple_trap    20    20       1.000
real           BSO            12     0       0.000
               GOx            12     5       0.417
               GOx_BSO        12    12       1.000
               control        12     0       0.000
               trap_xCTi      12    12       1.000
               triple_trap    12    12       1.000
               valve4         12    10       0.833
               valve4_GOx     12     6       0.500
synth          BSO            25     0       0.000
               GOx            25    10       0.400
               GOx_BSO        25    25       1.000
               control        25     0       0.000
               trap_xCTi      25    25       1.000
               triple_trap    25    25       1.000
               valve4         25    15       0.600
               valve4_GOx     25    12       0.480

## What the sample does show

Death rate per cohort and treatment condition, computed from the sample's `dead` column. Conditions fixed by construction (control, BSO, GOx_BSO, triple_trap, trap_xCTi) sit at 0% or 100%; the three conditions the benchmark uses (GOx, valve4, valve4_GOx) are the ones where the outcome depends on the cell profile.

In [4]:
summary = pd.read_csv(SAMPLE / "sample_outcomes_summary.csv")
check = (outcomes.groupby(["cohort", "condition"]).size().rename("rows_in_parquet").reset_index()
         .merge(summary, how="outer"))
check.head(20)

,cohort,condition,rows_in_parquet,n,n_dead,death_rate
0,normal_virtual,control,20,20,0,0.0000
1,normal_virtual,triple_trap,20,20,20,1.0000
2,real,BSO,12,12,0,0.0000
3,real,GOx,12,12,5,0.4167
4,real,GOx_BSO,12,12,12,1.0000
5,real,control,12,12,0,0.0000
6,real,trap_xCTi,12,12,12,1.0000
7,real,triple_trap,12,12,12,1.0000
8,real,valve4,12,12,10,0.8333
9,real,valve4_GOx,12,12,6,0.5000


The second table joins the shipped 18-row summary CSV to the counts recomputed from the Parquet file, as a consistency check between the two files of the sample.